# 해시 함수

해시 함수는 임의 크기 키 `k`를 **고정된 버킷 인덱스** `0 ~ m-1`로 옮긴다.

$$
h(k) \in \{0, 1, \dots, m-1\}
$$

좋은 해시 함수의 조건은 다음과 같다.

- **결정적**: 같은 키는 항상 같은 값
- **균등 분포**: 버킷이 한쪽으로 몰리지 않음
- **빠름**: 키 길이에 비례하는 시간
- **패턴이 버킷으로 새지 않음**: 연속 정수·비슷한 문자열이어도 인덱스가 한쪽으로 몰리지 않음

키가 `m`보다 많으면 충돌은 피할 수 없다(비둘기집 원리). 해시 테이블은 충돌을 체이닝·개방주소법으로 처리하고, 여기서는 **해시 함수 자체**만 본다.

자주 쓰는 구성은 아래 세 가지다.

| 방법 | 식 | 언제 |
| --- | --- | --- |
| 나눗셈법 | $k \bmod m$ | 정수 키, `m`이 소수 |
| 곱셈법 | $\lfloor m(kA \bmod 1) \rfloor$ | `m`이 2의 거듭제곱이어도 됨 |
| 문자열 해싱 | 다항식 롤링 해시 | 단어, 부분문자열 |

## 1. 나눗셈법 (Division method)

$$
h(k) = k \bmod m
$$

구현이 가장 단순하다. `m` 선택이 품질을 가른다.

- **소수를 고른다.** 키에 규칙이 있어도 나머지가 한쪽으로 덜 몰린다.
- **2의 거듭제곱은 피한다.** $m = 2^p$이면 $k \bmod m$은 `k`의 하위 `p`비트만 본다. 상위 비트에 담긴 정보가 버려진다.
- `m`이 키의 진법과 가까워도 치우친다. 예: 십진 키가 많은데 `m = 10`이면 끝자리만 본다.

테이블 크기 `m`을 소수로 두면 나눗셈법이 바로 실무에서 쓸 만한 수준이 된다.

In [1]:
def division_hash(key: int, m: int) -> int:
    """h(k) = k mod m. m은 보통 소수."""
    return key % m


m_prime = 13
m_pow2 = 16
keys = list(range(20, 52, 4))  # 20, 24, 28, ... 끝자리가 비슷한 키

print("key", "m=13(소수)", "m=16=2^4", sep="\t")
for k in keys:
    print(k, division_hash(k, m_prime), division_hash(k, m_pow2), sep="\t")

key	m=13(소수)	m=16=2^4
20	7	4
24	11	8
28	2	12
32	6	0
36	10	4
40	1	8
44	5	12
48	9	0


In [2]:
# m=16이면 하위 4비트만 남는다. 16으로 나눈 나머지 == k & 15
k = 0b_1010_0110  # 166
print(f"k        = {k:08b} ({k})")
print(f"k % 16   = {k % 16:08b} ({k % 16})")
print(f"k & 0b1111 = {k & 0b1111:08b} ({k & 0b1111})")

k        = 10100110 (166)
k % 16   = 00000110 (6)
k & 0b1111 = 00000110 (6)


## 2. 곱셈법 (Multiplication method)

$$
h(k) = \lfloor m \,(kA \bmod 1) \rfloor
$$

`kA`의 **소수부**만 남긴 뒤 `m`을 곱한다. Knuth는

$$
A \approx \frac{\sqrt{5}-1}{2} \approx 0.6180339887
$$

(황금비의 소수부)를 권한다. 이 `A`는 키가 연속이어도 소수부가 잘 섞인다.

나눗셈법과 다른 점:

- `m`이 2의 거듭제곱이어도 괜찮다. 소수부 전체에 키 비트가 섞인 다음, 앞쪽 비트만 버킷으로 쓴다.
- 소수 `m`을 고를 필요가 없어서 테이블 크기를 $2^r$로 두기 쉽다.

실수 곱은 큰 키에서 정밀도가 깨질 수 있다. `m = 2^r`이면 정수만으로 같은 식을 구현한다.

$$
h(k) = (A_{\text{int}} \cdot k \bmod 2^w) \;\gg\; (w-r)
$$

`w`는 워드 비트 수, $A_{\text{int}} = \lfloor 2^w A \rfloor$ 이다.

In [3]:
import math

GOLDEN_A = (math.sqrt(5) - 1) / 2  # ≈ 0.6180339887


def multiplication_hash(key: int, m: int, a: float = GOLDEN_A) -> int:
    """h(k) = floor(m * (kA mod 1)). 개념 확인용 실수 구현."""
    frac = (key * a) % 1.0
    return math.floor(m * frac)


def multiplication_hash_int(key: int, r: int, w: int = 32, a: int | None = None) -> int:
    """m = 2^r 일 때의 정수 구현. h(k) = ((A*k) mod 2^w) >> (w-r)"""
    if a is None:
        a = int((1 << w) * GOLDEN_A) | 1  # 홀수로 맞춤
    mask = (1 << w) - 1
    return ((a * key) & mask) >> (w - r)


m = 16  # 2^4
print("key", "실수곱", "정수비트", sep="\t")
for k in range(10):
    print(k, multiplication_hash(k, m), multiplication_hash_int(k, r=4), sep="\t")

key	실수곱	정수비트
0	0	0
1	9	9
2	3	3
3	13	13
4	7	7
5	1	1
6	11	11
7	5	5
8	15	15
9	8	8


In [8]:
from collections import Counter


def bucket_counts(hash_fn, keys, m: int) -> list[int]:
    counts = Counter(hash_fn(k) for k in keys)
    return [counts[i] for i in range(m)]


keys = list(range(0, 80, 4))  # 4의 배수만. 나눗셈법 + 2의거듭제곱에서 치우치기 쉬운 입력
m = 16

div_pow2 = bucket_counts(lambda k: division_hash(k, m), keys, m)
div_prime = bucket_counts(lambda k: division_hash(k, 13), keys, 13)
mul = bucket_counts(lambda k: multiplication_hash_int(k, r=4), keys, m)

print("나눗셈 m=16 (2^n) :", div_pow2)
print("나눗셈 m=13 (소수) :", div_prime)
print("곱셈   m=16 (2^n) :", mul)

나눗셈 m=16 (2^n) : [5, 0, 0, 0, 5, 0, 0, 0, 5, 0, 0, 0, 5, 0, 0, 0]
나눗셈 m=13 (소수) : [2, 1, 1, 2, 2, 1, 1, 2, 2, 1, 1, 2, 2]
곱셈   m=16 (2^n) : [2, 1, 1, 2, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 2]


## 3. 문자열 해싱 (Polynomial rolling hash)

문자를 정수로 바꾼 뒤 **진법처럼** 쌓는다. 밑 `p`, 모듈러 `mod`.

$$
h(s) = (s_0 p^{n-1} + s_1 p^{n-2} + \cdots + s_{n-1} p^0) \bmod \text{mod}
$$

Horner 규칙으로 한 패스에 계산한다.

$$
h \leftarrow 0,\quad h \leftarrow (h \cdot p + s_i) \bmod \text{mod}
$$

관례:

- `p`: 알파벳보다 큰 소수. 소문자만이면 **31**, 대소문자면 **53**
- `mod`: 큰 소수. $10^9+7$ 또는 $10^9+9$
- 문자 값: `'a' → 1`처럼 1부터 두면 `""`과 `"a"*0`이 섞이지 않는다

단순 합 `sum(ord(ch))`는 애너그램이 전부 충돌한다 (`"ab"`와 `"ba"`). 다항 해시는 순서가 바뀌면 값이 달라진다.

Python 내장 `hash()`는 프로세스마다 솔트가 달라서, 저장·제출용 해시로는 쓰지 않는다.

In [5]:
def poly_hash(s: str, p: int = 31, mod: int = 1_000_000_007) -> int:
    """다항식 롤링 해시. s는 소문자 알파벳 가정."""
    h = 0
    for ch in s:
        h = (h * p + (ord(ch) - ord("a") + 1)) % mod
    return h


def naive_sum_hash(s: str, mod: int = 1_000_000_007) -> int:
    return sum(ord(ch) - ord("a") + 1 for ch in s) % mod


words = ["ab", "ba", "abc", "acb", "aaa", "aab"]
print(f"{'s':<6} {'poly':>12} {'sum':>8}")
for w in words:
    print(f"{w:<6} {poly_hash(w):>12} {naive_sum_hash(w):>8}")

s              poly      sum
ab               33        3
ba               63        3
abc            1026        6
acb            1056        6
aaa             993        3
aab             994        4


### 롤링 해시

길이 `L`인 창을 한 칸 옮길 때, 맨 앞 문자를 빼고 새 문자를 붙이면 O(1)이다. Rabin-Karp의 핵심.

$$
h_{i+1} = \bigl( (h_i - s_i \, p^{L-1}) \cdot p + s_{i+L} \bigr) \bmod \text{mod}
$$

음수가 나오지 않게 모듈러를 한 번 더 보정한다.

In [6]:
def window_hashes(s: str, length: int, p: int = 31, mod: int = 1_000_000_007) -> list[int]:
    """길이 length인 모든 부분문자열의 다항 해시. 창 이동은 O(1)."""
    if length > len(s):
        return []

    p_pow = pow(p, length - 1, mod)  # 창 맨 앞 자리에 곱해진 p^(L-1)
    h = poly_hash(s[:length], p, mod)
    hashes = [h]

    for i in range(length, len(s)):
        left = ord(s[i - length]) - ord("a") + 1
        right = ord(s[i]) - ord("a") + 1
        h = (h - left * p_pow) % mod
        h = (h * p + right) % mod
        hashes.append(h)

    return hashes


text = "abacaba"
L = 3
got = window_hashes(text, L)
expected = [poly_hash(text[i : i + L]) for i in range(len(text) - L + 1)]

print("창     롤링      직접계산")
for i, (a, b) in enumerate(zip(got, expected)):
    print(f"{text[i:i+L]:<6} {a:<10} {b}")
print("일치:", got == expected)

창     롤링      직접계산
aba    1024       1024
bac    1956       1956
aca    1055       1055
cab    2916       2916
aba    1024       1024
일치: True


In [7]:
def find_pattern(text: str, pattern: str) -> list[int]:
    """롤링 해시로 pattern이 나타나는 시작 인덱스. 충돌 시 문자열을 한 번 더 비교."""
    n, m = len(text), len(pattern)
    if m == 0 or m > n:
        return []

    target = poly_hash(pattern)
    return [i for i, h in enumerate(window_hashes(text, m)) if h == target and text[i : i + m] == pattern]


print(find_pattern("abacaba", "aba"))  # [0, 4]
print(find_pattern("aaaaa", "aa"))     # [0, 1, 2, 3]

[0, 4]
[0, 1, 2, 3]


## 정리

| | 나눗셈법 | 곱셈법 | 문자열 다항 해시 |
| --- | --- | --- | --- |
| 식 | $k \bmod m$ | $\lfloor m(kA \bmod 1)\rfloor$ | $\sum s_i p^{n-1-i} \bmod \text{mod}$ |
| `m` | 소수가 안전 | $2^r$도 괜찮음 | 큰 소수 모듈러 |
| 장점 | 구현이 짧다 | 테이블 크기를 2의 거듭제곱으로 두기 쉽다 | 순서 구분, 창 이동 O(1) |
| 함정 | $m=2^p$면 하위 비트만 사용 | 실수곱은 큰 키에서 오차 | `p`/`mod`가 작으면 충돌 증가 |

코딩 테스트에서는 정수 키는 나눗셈법(또는 언어 딕셔너리), 문자열·부분문자열은 다항 롤링 해시를 가장 많이 쓴다. 곱셈법은 해시 테이블을 직접 구현할 때 `m = 2^r`로 잡고 비트 시프트로 인덱스를 뽑는 용도다.